In [2]:
import numpy as np


#  Code required to run sensitivity analysis - Please dont delete, 

In [ ]:
np.random.seed(42)
def patient_flow_one_day(t, l, patient_type):
    # Input: t: time, l: arrival rate for given ward, patient type: 1,2,3 corresponding to A, B or C
    # Output: patients for one date as: arrival time, patient type. 

    patients = []
    t_init = t
    while t < t_init + 1:
        wait =np.random.exponential(1/l)
        t += wait
        # Only record the patient if they arrived before the day ended
        if t < t_init + 1:
            patients.append([t, int(patient_type)])
        else:
            break 
    return patients


def patient_flow_year():
    #Generates the patient flow for the full year.
    #Output: patients for one the full year as: arrival time, patient type. 
    patient_type = np.array([1,2,3])
    t= 1

    patients_year = []

    while t < 365:
        patients_a_day =[]
        #Given Arrival rates
        lA = - (1/3650) * t**2 + (1/10)*t 
        lB = 1/5 * lA 
        lC = 6
        l = [lA, lB, lC]

        for i in range(len(patient_type)):
            patient_i_day_t =patient_flow_one_day(t, l[i], patient_type[i])
            patients_a_day.append(patient_i_day_t)
        t += 1
        patients_year.append(patients_a_day)

    return  patients_year


def ward_flow_year(init_dist, patient_flow):
    # Input: Init_dist: the initial distribution of beds as np.array([A,B,C]), patient_flow: the flow of paitents as arrival time, patient type for a year
    # Output: Amount of paritents that are relocated from ward, A, B and C, and the mean occupied beds for ward A, B and C.
    relocated_A= 0
    relocated_C = 0
    #Note: no one from B is truly relocated just admitted to A instead
    relocated_B= 0

    beds_occupied_A = []
    beds_occupied_B = []
    beds_occupied_C = []

    sigma2 = np.log(2)

    #Initializing all wards as empty beds corresponding to the initial distribution
    ward_A = np.zeros(init_dist[0])
    ward_B = np.zeros(init_dist[1])
    ward_C = np.zeros(init_dist[2])

    for t, type in patient_flow:
        #Release beds if time t has surpassed the length of the patients stay
        ward_A[ward_A <= t] = 0
        ward_B[ward_B <= t] = 0
        ward_C[ward_C <= t] = 0

        #List of indicies for where there is empty beds
        empty_beds_A = np.where(ward_A == 0)[0]
        empty_beds_B = np.where(ward_B == 0)[0]
        empty_beds_C = np.where(ward_C == 0)[0]

        beds_occupied_C.append(np.abs(len(empty_beds_C)-len(ward_C )))
        beds_occupied_B.append(np.abs(len(empty_beds_B)-len(ward_B )))
        beds_occupied_A.append(np.abs(len(empty_beds_A)-len(ward_A)))

        if type == 3:
            #LOS is length of stay
            mu = np.log(5*np.sqrt(2))
            LOS = np.random.lognormal(mu, sigma2)
            if len(empty_beds_C) > 0:
                #Takes the first bed in empty_beds and fills it with the time that patients stays.
                bed = empty_beds_C[0]
                ward_C[bed] = t + LOS
            else: 
                relocated_C += 1
    

        if type == 2:
            mu = np.log(6*np.sqrt(2))
            LOS = np.random.lognormal(mu, sigma2)

            if len(empty_beds_B) > 0:
                bed = empty_beds_B[0]
                ward_B[bed] = t + LOS

            #If there is no space in B we send patient to A
            elif len(empty_beds_A) > 0: 
                relocated_B += 1
                bed = empty_beds_A[0]
                ward_A[bed] = t + LOS
            else:
                #If A is occupied we discard a random patient from A and then admit to that bed.
                relocated_B += 1
                relocated_A += 1    
                bed = np.random.choice(len(ward_A))
                ward_A[bed] = t + LOS

        if type == 1:
            mu = np.log(4*np.sqrt(2))
            LOS = np.random.lognormal(mu, sigma2)

            if len(empty_beds_A) > 0:
                bed = empty_beds_A[0]
                ward_A[bed] = t + LOS
            else: 
                relocated_A += 1
        
        mean_occupied_A = np.mean(beds_occupied_A)
        mean_occupied_B = np.mean(beds_occupied_B)
        mean_occupied_C = np.mean(beds_occupied_C)
        
    return relocated_A, relocated_B, relocated_C, mean_occupied_A, mean_occupied_B, mean_occupied_C


def simulate_sum_n_years(init_dist,n):
    #Input: Init_dist: the initial distribution of beds as np.array([A,B,C]), n: Amount of times a year is simulated.
    A_discarted_lidt = []
    B_discarted_lidt = []
    C_discarted_lidt = []

    for _ in range(n):
        patients= patient_flow_year()
        #Flattens the nested structure 
        flat_patients = [p for block in patients for sublist in block for p in sublist]
        flat_patients_arr = np.array(flat_patients)

        discarded_A, discarded_B, discarded_C, mean_occupied_A, mean_occupied_B, mean_occupied_C  = ward_flow_year(init_dist, flat_patients)
        A_discarted_lidt.append(discarded_A)
        B_discarted_lidt.append(discarded_B)
        C_discarted_lidt.append(discarded_C)
    
    All = A_discarted_lidt + B_discarted_lidt + C_discarted_lidt

    return  np.mean(All)

    


# Sensitivity Analysis

We test the sensitivity to the distribution of 75, 80 and 100 total beds respectivly. The sensitivity measure is the sum of patients that are relocated from each ward. Including relocations from ward B to A, as well as relocations to other hospitals.

We generate X random guess and out put the best one, we do this X times to generate our initial guesses. ¨¨